# Exploración del `nairu_dataset.csv`

Análisis exploratorio del dataset consolidado del pipeline NAIRU Colombia.

**Objetivo:** caracterizar las series antes de modelar la NAIRU — coberturas, distribuciones, correlaciones, y verificación visual de relaciones macroeconómicas conocidas (Curva de Phillips, Ley de Okun).

**Convenciones:**
- Series mensuales: ``unemployment_rate``, ``tgp_rate``, ``ipc_index``, ``Inf_*``, ``brent_usd_per_barrel``, ``capacity_utilization``, ``TES_*``.
- Series trimestrales (NaN en meses 2,3,5,6,8,9,11,12): ``gap_viog_*``.
- Series anuales (NaN excepto enero): ``capital_stock_*``, ``human_capital``.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Localiza la raíz del proyecto desde el notebook
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET = ROOT / "data" / "final" / "nairu_dataset.csv"

df = pd.read_csv(DATASET, parse_dates=["date"]).set_index("date").sort_index()
print(f"Cargado: {DATASET}")
print(f"Filas: {len(df):,}  ·  Columnas: {len(df.columns)}")
print(f"Rango: {df.index.min().date()} → {df.index.max().date()}")
df.head()

## 1. Cobertura temporal por variable

Cada serie cubre un período distinto. Esta tabla muestra cuántas observaciones no-nulas hay y entre qué fechas. Útil para definir el subperíodo de modelado: el cuello de botella suele ser ``capacity_utilization`` (ANDI EOIC, desde 2017).

In [ ]:
rows = []
for col in df.columns:
    if col in ("year", "month"):
        continue
    s = df[col].dropna()
    rows.append({
        "variable": col,
        "obs": len(s),
        "inicio": s.index.min().date() if len(s) else None,
        "fin":    s.index.max().date() if len(s) else None,
        "mean":   round(s.mean(), 2) if len(s) else None,
        "std":    round(s.std(), 2)  if len(s) else None,
    })
coverage = pd.DataFrame(rows).set_index("variable")
coverage

## 2. Series de tiempo individuales

Una rejilla con cada variable mensual / trimestral en su propio panel para detectar visualmente quiebres, outliers y patrones estacionales.

In [ ]:
plot_cols = [
    "unemployment_rate", "tgp_rate", "informality_rate_13c",
    "Inf_Rate", "Inf_Goal", "Core_Inf",
    "brent_usd_per_barrel", "capacity_utilization",
    "TES_UVR_1Y", "TES_PESOS_1Y",
    "gap_viog_us", "gap_inv_viog_us",
]
plot_cols = [c for c in plot_cols if c in df.columns]

fig, axes = plt.subplots(
    nrows=(len(plot_cols) + 1) // 2, ncols=2,
    figsize=(14, 2.4 * len(plot_cols) // 2),
    sharex=True,
)
for ax, col in zip(axes.flat, plot_cols):
    df[col].dropna().plot(ax=ax, lw=1)
    ax.set_title(col, fontsize=10)
    ax.grid(alpha=0.3)
for ax in axes.flat[len(plot_cols):]:
    ax.axis("off")
plt.tight_layout()

## 3. Heatmap de correlaciones

Correlación de Pearson entre series mensuales en el período común a todas. Las fuertes correlaciones esperadas:

- ``unemployment_rate`` ↔ ``Inf_Rate``: negativa (Curva de Phillips).
- ``unemployment_rate`` ↔ ``capacity_utilization``: negativa (más ocupación → más uso de planta).
- ``Inf_Rate`` ↔ ``Inf_Goal``: positiva (la meta tiende a moverse con la observada).
- ``brent_usd_per_barrel`` ↔ ``Inf_Rate``: positiva con rezago (shock petróleo).

In [ ]:
monthly_cols = [
    "unemployment_rate", "tgp_rate", "informality_rate_13c",
    "Inf_Goal", "Inf_Rate", "Core_Inf",
    "brent_usd_per_barrel", "capacity_utilization",
    "TES_UVR_1Y", "TES_PESOS_1Y",
]
monthly_cols = [c for c in monthly_cols if c in df.columns]
corr = df[monthly_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(monthly_cols)))
ax.set_yticks(range(len(monthly_cols)))
ax.set_xticklabels(monthly_cols, rotation=60, ha="right")
ax.set_yticklabels(monthly_cols)
for i in range(len(monthly_cols)):
    for j in range(len(monthly_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i,j]) > 0.5 else "black",
                fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.04)
ax.set_title("Correlaciones (Pearson) — series mensuales")
plt.tight_layout()

## 4. Curva de Phillips visual

Relación inversa entre desempleo e inflación. Útil como **sanity check** antes de modelar — si no se ve la pendiente negativa, hay algo raro con las series.

Mostramos dos versiones:

1. Inflación total (``Inf_Rate``) vs desempleo.
2. Inflación núcleo (``Core_Inf``) vs desempleo — más limpia, menos ruido de shocks transitorios.

In [ ]:
phillips = df[["unemployment_rate", "Inf_Rate", "Core_Inf"]].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ycol, label in zip(
    axes,
    ["Inf_Rate", "Core_Inf"],
    ["Inflación total", "Inflación núcleo"],
):
    sub = phillips[["unemployment_rate", ycol]].dropna()
    ax.scatter(sub["unemployment_rate"], sub[ycol], alpha=0.4, s=15)
    # OLS para guía visual
    x, y = sub["unemployment_rate"], sub[ycol]
    if len(sub) > 5:
        b, a = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, a + b * xs, color="red", lw=1.2,
                label=f"y = {a:.2f} + {b:.3f}·x")
        ax.legend(loc="upper right")
    ax.set_xlabel("Tasa de desempleo (%)")
    ax.set_ylabel(f"{label} (%)")
    ax.set_title(f"Curva de Phillips — {label}")
    ax.grid(alpha=0.3)
plt.tight_layout()

## 5. Ley de Okun visual (aproximación)

La Ley de Okun relaciona la **brecha del producto** con la **brecha del desempleo**. Como aún no tenemos VIOG-Colombia, usamos como proxy ``gap_viog_us`` (brecha de USA) o ``Δunemployment_rate`` (cambio en desempleo).

Esta gráfica se completará con sentido económico cuando exista ``gap_viog_co``.

In [ ]:
# Dado que VIOG es trimestral, hacemos resample mensual hacia adelante
okun_pre = df[["unemployment_rate", "gap_viog_us"]].copy()
okun_pre["gap_viog_us"] = okun_pre["gap_viog_us"].ffill(limit=2)
okun = okun_pre.dropna()

if len(okun) > 5:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(okun["unemployment_rate"], okun["gap_viog_us"], alpha=0.4, s=15)
    x, y = okun["unemployment_rate"], okun["gap_viog_us"]
    b, a = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, a + b * xs, color="red", lw=1.2,
            label=f"y = {a:.3f} + {b:.4f}·x")
    ax.set_xlabel("Desempleo Colombia (%)")
    ax.set_ylabel("Brecha del producto USA (gap_viog_us)")
    ax.set_title("Aproximación a Okun (placeholder — usar gap_viog_co cuando esté disponible)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
else:
    print("Sin datos suficientes para graficar Okun en el período común.")

## 6. Inflación interanual desde IPC

El roadmap menciona que falta calcular la **inflación interanual** desde el IPC. Hagamos un quick check: si BANREP publica ``Inf_Rate`` como inflación interanual, debería coincidir aproximadamente con ``ipc.pct_change(periods=12) * 100``.

In [ ]:
ipc = df["ipc_index"].dropna()
inf_yoy_calc = ipc.pct_change(periods=12) * 100

comparison = pd.concat([
    inf_yoy_calc.rename("inflation_yoy_from_ipc"),
    df["Inf_Rate"].rename("Inf_Rate (BANREP)"),
], axis=1).dropna()

if len(comparison) > 0:
    fig, ax = plt.subplots(figsize=(11, 4))
    comparison.plot(ax=ax, lw=1)
    ax.set_title("Inflación interanual: BANREP vs cálculo desde IPC DANE")
    ax.set_ylabel("%")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    diff = (comparison.iloc[:, 0] - comparison.iloc[:, 1]).abs()
    print(f"Diferencia absoluta promedio: {diff.mean():.3f} pp")
    print(f"Máxima diferencia:           {diff.max():.3f} pp")

---

## Pendientes para iteraciones futuras

- Reemplazar `gap_viog_us` por `gap_viog_co` cuando se reciba el PIB potencial Colombia (input externo).
- Modelar la NAIRU (tarea fuera de este repo — código provisto por compañero).
- Comparar la base con `data/inputs/nairu_estimates_v6.csv` → ver `validation_v6.ipynb`.